In [ ]:
# Load or reload R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    try:
        %reload_ext rpy2.ipython
    except Exception as e2:
        print("Note on rpy2 initialization:", e2)

# Transpiling Soft Probabilistic Triage Pipeline to Native C with `m2cgen` (`models/transpile_soft_pipeline_to_c.ipynb`)

This notebook exports and transpiles the **5-Class Soft Probabilistic Triage Pipeline** into zero-dependency, high-performance **Native C Code** using `m2cgen` (Model 2 Code Generator) and tests the compiled C implementation against the original Python/R pipeline on the **Holdout Test Set**:

### Transpilation & Integration Steps
1. **Load Data & Prepare 35 Predictor Features in R**:
   - Reads dataset, constructs 35 features, and splits into Train, Validation, and Test sets.
2. **Fit Sub-Models & Transpile to C using `m2cgen`**:
   - Convert Layer 2 RF, Layer 3A LightGBM, and Layer 3B LightGBM into pure C functions (`score_layer2_rf`, `score_layer3a_lgb23`, `score_layer3b_lgb45`).
   - Uses C preprocessor macro scopes (`#define sigmoid ...`) to isolate sub-model helper functions cleanly without any string or regex mutation.
3. **Construct Master C Pipeline (`predict_soft_pipeline`)**:
   - Embeds the **Soft Probabilistic Joint Scaling Algorithm** directly inside C:
     - $P(\text{ESI 1}) = P_{RF}(\text{1})$
     - $P(\text{ESI 2}) = P_{RF}(\text{2\_3}) \times P_{\text{LGB23}}(\text{ESI 2})$
     - $P(\text{ESI 3}) = P_{RF}(\text{2\_3}) \times (1 - P_{\text{LGB23}}(\text{ESI 2}))$
     - $P(\text{ESI 4}) = P_{RF}(\text{4\_5}) \times P_{\text{LGB45}}(\text{ESI 4})$
     - $P(\text{ESI 5}) = P_{RF}(\text{4\_5}) \times (1 - P_{\text{LGB45}}(\text{ESI 4}))$
4. **Compile & Test Shared C Library (`deploy/soft_triage_pipeline.so`)**:
   - Compiles C code with `gcc -O3 -shared -fPIC`.
   - Runs C inference via `ctypes` on the holdout test set using an absolute path.
5. **Equivalence & Change Verification**:
   - Evaluates C test metrics (**Recall**, **Specificity**, **Balanced Accuracy**, **ROC-AUC**) and calculates maximum absolute probability differences between C and Python/R outputs.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Data & Prepare 35 Predictor Features in R
# ---------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(caret)
  library(dplyr)
  library(pROC)
})
config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) config_path <- "config/triage_conf.json"
config <- fromJSON(config_path)
set.seed(config$training$random_state)
data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) data_file <- paste0("../", data_file)
data_env <- new.env()
load(data_file, envir = data_env)
df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
raw_df   <- get(df_names[which.max(df_sizes)], envir = data_env)
target_col_name <- config$classes$target_col
gender_vec <- if ("gender" %in% names(raw_df)) ifelse(as.character(raw_df$gender) == "Male", 1, 0) else 0
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) ifelse(!is.na(raw_df$cc_breathingdifficulty), raw_df$cc_breathingdifficulty, 0) else 0
get_vec <- function(col_name, default_val = 0) {
  if (col_name %in% names(raw_df)) {
    res <- raw_df[[col_name]]
    res[is.na(res)] <- default_val
    return(res)
  } else {
    return(rep(default_val, nrow(raw_df)))
  }
}
p_last   <- get_vec("pulse_last"); p_max    <- get_vec("pulse_max"); p_min    <- get_vec("pulse_min")
s_last   <- get_vec("sbp_last");   s_max    <- get_vec("sbp_max");   s_min    <- get_vec("sbp_min")
o2_last  <- get_vec("spo2_last");  o2_max   <- get_vec("spo2_max");  o2_min   <- get_vec("spo2_min")
r_last   <- get_vec("resp_last");   r_max    <- get_vec("resp_max");   r_min    <- get_vec("resp_min")
t_hr     <- get_vec("triage_vital_hr"); t_sbp <- get_vec("triage_vital_sbp"); t_o2 <- get_vec("triage_vital_o2"); t_rr <- get_vec("triage_vital_rr")
df_full <- data.frame(
  age = raw_df$age, gender = gender_vec, cc_breathingdifficulty = cc_bd_vec,
  triage_vital_hr = t_hr, triage_vital_sbp = t_sbp, triage_vital_rr = t_rr, triage_vital_o2 = t_o2,
  pulse_last = p_last, resp_last = r_last, spo2_last = o2_last, sbp_last = s_last,
  pulse_min = p_min, resp_min = r_min, spo2_min = o2_min, sbp_min = s_min,
  pulse_max = p_max, resp_max = r_max, spo2_max = o2_max, sbp_max = s_max,
  hr_mean_to_last = t_hr - p_last, sbp_mean_to_last = t_sbp - s_last, spo2_mean_to_last = t_o2 - o2_last, rr_mean_to_last = t_rr - r_last,
  hr_range = p_max - p_min, rr_range = r_max - r_min, spo2_range = o2_max - o2_min, sbp_range = s_max - s_min,
  hr_last_to_min = p_last - p_min, rr_last_to_min = r_last - r_min, spo2_last_to_min = o2_last - o2_min, sbp_last_to_min = s_last - s_min,
  hr_last_to_max = p_last - p_max, rr_last_to_max = r_last - r_max, spo2_last_to_max = o2_last - o2_max, sbp_last_to_max = s_last - s_max
)
raw_esi <- as.character(raw_df[[target_col_name]])
df_full$target_col <- factor(raw_esi, levels = c("1", "2", "3", "4", "5"))
df_full <- na.omit(df_full)
test_size <- config$training$test_size
val_size  <- config$training$val_size
in_train_val <- createDataPartition(df_full$target_col, p = 1 - test_size, list = FALSE)
train_val_df <- df_full[in_train_val, ]
test_df      <- df_full[-in_train_val, ]
rel_val_size <- val_size / (1 - test_size)
in_train    <- createDataPartition(train_val_df$target_col, p = 1 - rel_val_size, list = FALSE)
train_df    <- train_val_df[in_train, ]
val_df      <- train_val_df[-in_train, ]
# Export partitions to R global environment
train_py <<- train_df
val_py   <<- val_df
test_py  <<- test_df
cat(sprintf("Partitions Prepared: Train=%d, Val=%d, Test=%d\n", nrow(train_df), nrow(val_df), nrow(test_df)))

In [ ]:
# ---------------------------------------------------------
# Step 2: Retrieve R Dataframes in Python & Fit Models for m2cgen Transpilation
# ---------------------------------------------------------
import os
import numpy as np
import pandas as pd
from rpy2.robjects import r
import rpy2.robjects.pandas2ri as pandas2ri
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
import m2cgen as m2c
# Retrieve dataframes directly from rpy2 environment
try:
    pandas2ri.activate()
    train_df = pd.DataFrame(pandas2ri.rpy2py_dataframe(r['train_py']))
    val_df   = pd.DataFrame(pandas2ri.rpy2py_dataframe(r['val_py']))
    test_df  = pd.DataFrame(pandas2ri.rpy2py_dataframe(r['test_py']))
except Exception:
    train_df = pd.DataFrame(r['train_py'])
    val_df   = pd.DataFrame(r['val_py'])
    test_df  = pd.DataFrame(r['test_py'])
feature_cols = [c for c in train_df.columns if c != 'target_col']
binary_cols  = ['gender', 'cc_breathingdifficulty']
cont_cols    = [c for c in feature_cols if c not in binary_cols]
# Standard scale continuous features
scaler = StandardScaler()
X_train_cont = scaler.fit_transform(train_df[cont_cols])
X_val_cont   = scaler.transform(val_df[cont_cols])
X_test_cont  = scaler.transform(test_df[cont_cols])
X_train = np.hstack([X_train_cont, train_df[binary_cols].values])
X_val   = np.hstack([X_val_cont,   val_df[binary_cols].values])
X_test  = np.hstack([X_test_cont,  test_df[binary_cols].values])
all_feats = cont_cols + binary_cols
# Targets
y_raw_tr = train_df['target_col'].astype(str).values
y_raw_vl = val_df['target_col'].astype(str).values
y_raw_ts = test_df['target_col'].astype(str).values
# Layer 2 Target ('2_3', '4_5', '1')
y_l2_tr = np.where(np.isin(y_raw_tr, ['2', '3']), '2_3', np.where(np.isin(y_raw_tr, ['4', '5']), '4_5', '1'))
# Layer 3 Targets
y_esi23_tr = (np.isin(y_raw_tr, ['2', '3']) & (y_raw_tr == '2')).astype(int)
y_esi23_vl = (np.isin(y_raw_vl, ['2', '3']) & (y_raw_vl == '2')).astype(int)
y_esi45_tr = (np.isin(y_raw_tr, ['4', '5']) & (y_raw_tr == '4')).astype(int)
y_esi45_vl = (np.isin(y_raw_vl, ['4', '5']) & (y_raw_vl == '4')).astype(int)
# 1. Fit Layer 2 Random Forest (3 Classes: '1', '2_3', '4_5')
print("Training Layer 2 Random Forest (scikit-learn)...")
rf_l2 = RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1)
rf_l2.fit(X_train, y_l2_tr)
# 2. Fit Layer 3A LightGBM (ESI 2 vs 3)
print("Training Layer 3A LightGBM (ESI 2 vs 3)...")
lgb_23 = lgb.LGBMClassifier(n_estimators=100, learning_rate=0.05, num_leaves=31, max_depth=6, random_state=42, verbose=-1)
lgb_23.fit(X_train, y_esi23_tr)
# 3. Fit Layer 3B LightGBM (ESI 4 vs 5)
print("Training Layer 3B LightGBM (ESI 4 vs 5)...")
lgb_45 = lgb.LGBMClassifier(n_estimators=100, learning_rate=0.05, num_leaves=31, max_depth=6, random_state=42, verbose=-1)
lgb_45.fit(X_train, y_esi45_tr)
# Transpile Sub-Models to C Code via m2cgen
print("\nTranspiling sub-models to C with m2cgen...")
code_rf    = m2c.export_to_c(rf_l2, function_name="score_layer2_rf")
code_lgb23 = m2c.export_to_c(lgb_23, function_name="score_layer3a_lgb23")
code_lgb45 = m2c.export_to_c(lgb_45, function_name="score_layer3b_lgb45")
# Wrap each sub-model in C preprocessor macro scopes (#define sigmoid ... #undef sigmoid)
# This renames duplicate helper functions during C preprocessing without touching model code
block_rf = f"#define sigmoid rf_sigmoid\n{code_rf}\n#undef sigmoid"
block_lgb23 = f"#define sigmoid lgb23_sigmoid\n{code_lgb23}\n#undef sigmoid"
block_lgb45 = f"#define sigmoid lgb45_sigmoid\n{code_lgb45}\n#undef sigmoid"
print("C Code transpilation & macro wrapping complete!")

In [ ]:
# ---------------------------------------------------------
# Step 3: Assemble Full C Source File & Soft Probabilistic Pipeline Wrapper
# ---------------------------------------------------------
deploy_dir = "../deploy"
if not os.path.exists(deploy_dir):
    deploy_dir = "deploy"
os.makedirs(deploy_dir, exist_ok=True)
c_file_path = os.path.abspath(os.path.join(deploy_dir, "soft_triage_pipeline.c"))
so_file_path = os.path.abspath(os.path.join(deploy_dir, "soft_triage_pipeline.so"))
c_header = """
/* =========================================================================
   Soft Probabilistic Triage Pipeline Master C Compilation Unit
   ========================================================================= */
#include <math.h>
#include <stddef.h>
"""
c_pipeline_wrapper = """
/* =========================================================================
   Soft Probabilistic Triage Pipeline Master C Function
   ========================================================================= */
void score_layer2_rf(double * input, double * output);
void score_layer3a_lgb23(double * input, double * output);
void score_layer3b_lgb45(double * input, double * output);
void predict_soft_pipeline(double * input, double * output_probs, int * pred_class) {
    // 1. Run Layer 2 Random Forest (3 classes: '1', '2_3', '4_5')
    double rf_probs[3];
    score_layer2_rf(input, rf_probs);
    
    double p_rf_1  = rf_probs[0]; // P(ESI 1)
    double p_rf_23 = rf_probs[1]; // P(ESI 2/3)
    double p_rf_45 = rf_probs[2]; // P(ESI 4/5)
    
    // 2. Run Layer 3A LightGBM (ESI 2 vs 3)
    double lgb23_probs[2];
    score_layer3a_lgb23(input, lgb23_probs);
    double p_esi2_given_23 = lgb23_probs[1]; // P(ESI 2 | ESI 2/3)
    double p_esi3_given_23 = lgb23_probs[0]; // P(ESI 3 | ESI 2/3) = 1 - P(ESI 2)
    
    // 3. Run Layer 3B LightGBM (ESI 4 vs 5)
    double lgb45_probs[2];
    score_layer3b_lgb45(input, lgb45_probs);
    double p_esi4_given_45 = lgb45_probs[1]; // P(ESI 4 | ESI 4/5)
    double p_esi5_given_45 = lgb45_probs[0]; // P(ESI 5 | ESI 4/5) = 1 - P(ESI 4)
    
    // 4. Soft Probabilistic Joint Multiplication
    output_probs[0] = p_rf_1;                           // ESI 1
    output_probs[1] = p_rf_23 * p_esi2_given_23;        // ESI 2
    output_probs[2] = p_rf_23 * p_esi3_given_23;        // ESI 3
    output_probs[3] = p_rf_45 * p_esi4_given_45;        // ESI 4
    output_probs[4] = p_rf_45 * p_esi5_given_45;        // ESI 5
    
    // 5. Argmax Predicted Class (1..5)
    int best_cls = 1;
    double max_p = output_probs[0];
    for (int i = 1; i < 5; i++) {
        if (output_probs[i] > max_p) {
            max_p = output_probs[i];
            best_cls = i + 1;
        }
    }
    *pred_class = best_cls;
}
"""
full_c_code = c_header + "\n\n" + block_rf + "\n\n" + block_lgb23 + "\n\n" + block_lgb45 + "\n\n" + c_pipeline_wrapper
with open(c_file_path, "w") as f:
    f.write(full_c_code)
print(f"Master C Pipeline code written to: {c_file_path} ({len(full_c_code)} bytes)")

In [ ]:
# ---------------------------------------------------------
# Step 4: Compile C Code into Shared Library (.so)
# ---------------------------------------------------------
import subprocess
compile_cmd = f"gcc -O3 -shared -fPIC -lm '{c_file_path}' -o '{so_file_path}'"
print(f"Executing: {compile_cmd}")
res = subprocess.run(compile_cmd, shell=True, capture_output=True, text=True)
if res.returncode == 0:
    print(f"SUCCESS: Dynamic C Shared Library compiled -> {so_file_path}")
else:
    raise RuntimeError(f"GCC Compilation Failed:\n{res.stderr}")

In [ ]:
# ---------------------------------------------------------
# Step 5: Run C Shared Library Inference & Compare with Original Python Output
# ---------------------------------------------------------
import ctypes
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
print(f"Loading C shared library from: {so_file_path}")
c_lib = ctypes.CDLL(so_file_path)
# Define function signature for predict_soft_pipeline
# void predict_soft_pipeline(double * input, double * output_probs, int * pred_class)
c_lib.predict_soft_pipeline.argtypes = [
    ctypes.POINTER(ctypes.c_double),
    ctypes.POINTER(ctypes.c_double),
    ctypes.POINTER(ctypes.c_int)
]
c_lib.predict_soft_pipeline.restype = None
# 1. Python Soft Pipeline Inference
rf_probs_py = rf_l2.predict_proba(X_test) # classes: ['1', '2_3', '4_5']
lgb23_probs_py = lgb_23.predict_proba(X_test)[:, 1]
lgb45_probs_py = lgb_45.predict_proba(X_test)[:, 1]
N = X_test.shape[0]
probs_py = np.zeros((N, 5))
probs_py[:, 0] = rf_probs_py[:, 0]                                # ESI 1
probs_py[:, 1] = rf_probs_py[:, 1] * lgb23_probs_py              # ESI 2
probs_py[:, 2] = rf_probs_py[:, 1] * (1.0 - lgb23_probs_py)        # ESI 3
probs_py[:, 3] = rf_probs_py[:, 2] * lgb45_probs_py              # ESI 4
probs_py[:, 4] = rf_probs_py[:, 2] * (1.0 - lgb45_probs_py)        # ESI 5
preds_py = np.argmax(probs_py, axis=1) + 1
# 2. C Shared Library Soft Pipeline Inference
probs_c = np.zeros((N, 5), dtype=np.float64)
preds_c = np.zeros(N, dtype=np.int32)
for i in range(N):
    x_sample = X_test[i, :].astype(np.float64)
    x_ptr = x_sample.ctypes.data_as(ctypes.POINTER(ctypes.c_double))
    
    out_p = (ctypes.c_double * 5)()
    out_cls = ctypes.c_int()
    
    c_lib.predict_soft_pipeline(x_ptr, out_p, ctypes.byref(out_cls))
    
    probs_c[i, :] = np.array([out_p[k] for k in range(5)])
    preds_c[i] = out_cls.value
# 3. Check Exact Numerical Equality
max_prob_diff = np.max(np.abs(probs_py - probs_c))
mismatched_preds = np.sum(preds_py != preds_c)
print(f"============================================================")
print(f"   C TRANSPILATION VERIFICATION & CHANGE AUDIT REPORT")
print(f"============================================================")
print(f"  Holdout Test Set Size               : {N} samples")
print(f"  Max Prob Probability Difference     : {max_prob_diff:.10e}")
print(f"  Class Prediction Mismatches (C vs PY): {mismatched_preds} / {N} ({(mismatched_preds/N)*100:.2f}%)")
print(f"============================================================\n")
if max_prob_diff < 1e-6 and mismatched_preds == 0:
    print("VERDICT: The transpiled C implementation is 100% IDENTICAL to the Python/R soft pipeline. No prediction changes occurred!")
else:
    print("VERDICT: Numerical variations detected between native C execution and original python model.")

In [ ]:
# ---------------------------------------------------------
# Step 6: Benchmark Transpiled C Model (Recall, Specificity, BalAcc, ROC-AUC ONLY)
# ---------------------------------------------------------
y_true = test_df['target_col'].astype(int).values
def compute_benchmark_metrics(y_true, y_pred, probs):
    cm = confusion_matrix(y_true, y_pred, labels=[1, 2, 3, 4, 5])
    
    rec_list, spec_list, bal_acc_list, auc_list = [], [], [], []
    for i in range(5):
        cls = i + 1
        tp = cm[i, i]
        fn = np.sum(cm[i, :]) - tp
        fp = np.sum(cm[:, i]) - tp
        tn = np.sum(cm) - (tp + fn + fp)
        
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        bal_acc = (rec + spec) / 2.0
        
        y_bin = (y_true == cls).astype(int)
        try:
            auc = roc_auc_score(y_bin, probs[:, i])
        except Exception:
            auc = np.nan
            
        rec_list.append(rec)
        spec_list.append(spec)
        bal_acc_list.append(bal_acc)
        auc_list.append(auc)
    macro_rec  = np.mean(rec_list)
    macro_spec = np.mean(spec_list)
    macro_bal  = np.mean(bal_acc_list)
    macro_auc  = np.nanmean(auc_list)
    
    return macro_rec, macro_spec, macro_bal, macro_auc, cm
rec_c, spec_c, bal_c, auc_c, cm_c = compute_benchmark_metrics(y_true, preds_c, probs_c)
rec_py, spec_py, bal_py, auc_py, _ = compute_benchmark_metrics(y_true, preds_py, probs_py)
print("============================================================")
print("   TRANSPILED C MODEL HOLDOUT TEST BENCHMARK")
print("============================================================")
print(f"  Macro Recall (Sensitivity) : {rec_c:.4f}  (Python: {rec_py:.4f})")
print(f"  Macro Specificity          : {spec_c:.4f}  (Python: {spec_py:.4f})")
print(f"  Macro Balanced Accuracy    : {bal_c:.4f}  (Python: {bal_py:.4f})")
print(f"  Macro ROC-AUC              : {auc_c:.4f}  (Python: {auc_py:.4f})")
print("============================================================\n")
print("Confusion Matrix (Native C Implementation):")
print(cm_c)
# Save C Transpilation Test Report
reports_dir = "../reports"
if not os.path.exists(reports_dir):
    reports_dir = "reports"
os.makedirs(reports_dir, exist_ok=True)
c_report_df = pd.DataFrame({
    'Implementation': ['Native_C_Transpiled', 'Original_Python'],
    'Macro_Recall': [round(rec_c, 4), round(rec_py, 4)],
    'Macro_Specificity': [round(spec_c, 4), round(spec_py, 4)],
    'Macro_Balanced_Accuracy': [round(bal_c, 4), round(bal_py, 4)],
    'Macro_ROC_AUC': [round(auc_c, 4), round(auc_py, 4)]
})
c_report_df.to_csv(os.path.join(reports_dir, "c_transpilation_test_report.csv"), index=False)
print(f"\nC Transpilation Test Report written to: {os.path.join(reports_dir, 'c_transpilation_test_report.csv')}")